# Project 1 — RDKit Molecule Analyser

This notebook is a detailed first-principles RDKit practice project. It rebuilds the original Kaggle `dl-chem` Project 1 idea without trimming the learning steps: parse SMILES, create RDKit molecule objects, draw structures, calculate molecular descriptors, check Lipinski-style rules, detect functional groups with SMARTS, analyse molecule lists and export clean results.

**Core workflow**

```text
SMILES string → RDKit Mol object → structure image + descriptors + rule checks + functional-group labels
```

This project is intentionally simple but important: Projects 2–5 reuse the same ideas for descriptor-based QSAR, Morgan fingerprints, molecular classification and deep learning chemistry.

## 0. Environment notes

Recommended environment:

```bash
conda install -c conda-forge rdkit pandas matplotlib jupyter
```

or, in a temporary Kaggle/Colab notebook:

```bash
pip install rdkit
```

For a GitHub practice repo, keep installation commands in `README.md` or `environment.yml` rather than executing `pip install` inside every notebook.

In [ ]:
# ============================================================
# Part 1 — Imports
# ============================================================

from pathlib import Path

import pandas as pd
from IPython.display import display

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, Crippen, Lipinski, rdMolDescriptors

## 1. Start with one SMILES string

A **SMILES** string is a compact text representation of a molecule. RDKit converts SMILES into a `Mol` object that can be used for drawing, descriptor calculation and substructure matching.

Example:

```text
CCO = ethanol
```

The key function is:

```python
Chem.MolFromSmiles(smiles)
```

If the SMILES is invalid, RDKit returns `None`.

In [ ]:
# ============================================================
# Part 2 — Parse one molecule from SMILES
# ============================================================

smiles = "CCO"  # ethanol

mol = Chem.MolFromSmiles(smiles)

if mol is None:
    raise ValueError(f"Invalid SMILES: {smiles}")

print("Valid molecule")
print("Input SMILES:", smiles)
print("Canonical SMILES:", Chem.MolToSmiles(mol, canonical=True))
print("Number of atoms:", mol.GetNumAtoms())
print("Number of bonds:", mol.GetNumBonds())

## 2. Draw the molecule

`Draw.MolToImage()` converts an RDKit molecule into a 2D structure image. This is useful for sanity-checking whether the SMILES string represents the molecule you intended.

In [ ]:
# ============================================================
# Part 3 — Draw one molecule
# ============================================================

img = Draw.MolToImage(mol, size=(300, 300))
display(img)

## 3. Calculate basic molecular properties

Project 1 calculates the same descriptors that are commonly reused in QSAR and cheminformatics workflows.

| Descriptor | Meaning | Why it matters |
|---|---|---|
| Molecular weight | Molecular mass | Size, volatility, permeability and drug-likeness |
| LogP | Octanol/water partition tendency | Hydrophobicity and lipophilicity |
| H-bond donors | Number of H-donor groups | Polarity and binding/solubility behaviour |
| H-bond acceptors | Number of H-acceptor atoms | Polarity and interaction capacity |
| TPSA | Topological polar surface area | Polarity and permeability proxy |
| Rotatable bonds | Flexible single bonds | Molecular flexibility |
| Ring count | Number of ring systems | Structural rigidity |
| Aromatic ring count | Aromatic ring systems | Common in drugs, solvents and fragrance molecules |

In [ ]:
# ============================================================
# Part 4 — Basic molecular descriptors
# ============================================================

mw = Descriptors.MolWt(mol)
logp = Crippen.MolLogP(mol)
h_donors = Lipinski.NumHDonors(mol)
h_acceptors = Lipinski.NumHAcceptors(mol)
tpsa = rdMolDescriptors.CalcTPSA(mol)
rot_bonds = Lipinski.NumRotatableBonds(mol)
ring_count = Lipinski.RingCount(mol)
aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
heavy_atoms = Lipinski.HeavyAtomCount(mol)

print("Molecular weight:", round(mw, 3))
print("LogP:", round(logp, 3))
print("H-bond donors:", h_donors)
print("H-bond acceptors:", h_acceptors)
print("TPSA:", round(tpsa, 3))
print("Rotatable bonds:", rot_bonds)
print("Ring count:", ring_count)
print("Aromatic rings:", aromatic_rings)
print("Heavy atoms:", heavy_atoms)

## 4. Store results in a table

A single molecule can be stored as one row in a `pandas.DataFrame`. This is the bridge from chemistry to machine learning: once descriptors are in a table, they can become model features.

In [ ]:
# ============================================================
# Part 5 — Store descriptor results in a DataFrame
# ============================================================

single_result = {
    "name": "ethanol",
    "smiles": smiles,
    "canonical_smiles": Chem.MolToSmiles(mol, canonical=True),
    "MolWt": round(mw, 3),
    "LogP": round(logp, 3),
    "HBD": h_donors,
    "HBA": h_acceptors,
    "TPSA": round(tpsa, 3),
    "RotatableBonds": rot_bonds,
    "RingCount": ring_count,
    "AromaticRings": aromatic_rings,
    "HeavyAtomCount": heavy_atoms,
}

single_df = pd.DataFrame([single_result])
single_df

## 5. Lipinski Rule of Five

The Lipinski Rule of Five is a rough oral drug-likeness filter. It is not a universal law and should not be used blindly, but it is a useful beginner example of rule-based molecular filtering.

Common thresholds:

| Rule | Pass condition |
|---|---|
| Molecular weight | ≤ 500 |
| LogP | ≤ 5 |
| Hydrogen bond donors | ≤ 5 |
| Hydrogen bond acceptors | ≤ 10 |

A molecule passing all four rules is often described as more oral-drug-like, but many real drugs and natural products violate one or more rules.

In [ ]:
# ============================================================
# Part 6 — Lipinski Rule-of-Five check
# ============================================================

def lipinski_check(mol):
    """Return Lipinski rule pass/fail values and total number of passed rules."""

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    h_donors = Lipinski.NumHDonors(mol)
    h_acceptors = Lipinski.NumHAcceptors(mol)

    rules = {
        "MW <= 500": mw <= 500,
        "LogP <= 5": logp <= 5,
        "HBD <= 5": h_donors <= 5,
        "HBA <= 10": h_acceptors <= 10,
    }

    passed = sum(rules.values())
    return rules, passed


rules, passed = lipinski_check(mol)

print("Lipinski rules passed:", passed, "/ 4")
for rule, result in rules.items():
    print(f"{rule}: {result}")

## 6. Functional group detection using SMARTS

SMARTS is a pattern language for substructure searching. It is similar to SMILES, but it describes molecular patterns rather than exact molecules.

This section uses a small functional-group dictionary. It is not exhaustive, but it demonstrates the core idea:

```text
functional group SMARTS pattern → RDKit substructure match → detected group name
```

In [ ]:
# ============================================================
# Part 7 — Functional group SMARTS dictionary
# ============================================================

functional_groups = {
    "Alcohol": "[OX2H]",
    "Carboxylic acid": "C(=O)[OH]",
    "Aldehyde": "[CX3H1](=O)",
    "Ketone": "[#6][CX3](=O)[#6]",
    "Ester": "C(=O)O[#6]",
    "Ether": "[OD2]([#6])[#6]",
    "Amine": "[NX3;H2,H1,H0]",
    "Amide": "C(=O)N",
    "Benzene ring": "c1ccccc1",
    "Alkene": "C=C",
    "Alkyne": "C#C",
    "Phenol": "c[OX2H]",
    "Nitrile": "C#N",
    "Halide": "[F,Cl,Br,I]",
}

compiled_functional_groups = {
    name: Chem.MolFromSmarts(smarts)
    for name, smarts in functional_groups.items()
}

# Validate that all SMARTS patterns compiled successfully.
invalid_patterns = [
    name for name, pattern in compiled_functional_groups.items()
    if pattern is None
]

if invalid_patterns:
    raise ValueError(f"Invalid SMARTS patterns: {invalid_patterns}")

print("Functional-group patterns loaded:", len(compiled_functional_groups))

In [ ]:
# ============================================================
# Part 8 — Functional group detector
# ============================================================

def detect_functional_groups(mol):
    """Detect functional groups using predefined SMARTS patterns."""

    found = []

    for name, pattern in compiled_functional_groups.items():
        if mol.HasSubstructMatch(pattern):
            found.append(name)

    return found


found_groups = detect_functional_groups(mol)

print("Functional groups found:")
if found_groups:
    for group in found_groups:
        print("-", group)
else:
    print("No listed functional groups found")

## 7. Build one reusable molecule analyser

This function combines the previous steps into one workflow:

```text
SMILES → validation → canonical SMILES → descriptors → Lipinski check → functional groups → optional image
```

This is the first reusable function in the repo and becomes the foundation for later projects.

In [ ]:
# ============================================================
# Part 9 — Reusable molecule analyser
# ============================================================

def analyse_molecule(smiles, name=None, show_molecule=True):
    """Analyse one molecule from a SMILES string and return a one-row DataFrame."""

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None

    canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

    mw = Descriptors.MolWt(mol)
    logp = Crippen.MolLogP(mol)
    h_donors = Lipinski.NumHDonors(mol)
    h_acceptors = Lipinski.NumHAcceptors(mol)
    tpsa = rdMolDescriptors.CalcTPSA(mol)
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    ring_count = Lipinski.RingCount(mol)
    aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)
    heavy_atoms = Lipinski.HeavyAtomCount(mol)
    fraction_csp3 = rdMolDescriptors.CalcFractionCSP3(mol)

    rules, passed = lipinski_check(mol)
    found_groups = detect_functional_groups(mol)

    result = {
        "name": name if name is not None else "",
        "smiles": smiles,
        "canonical_smiles": canonical_smiles,
        "MolWt": round(mw, 3),
        "LogP": round(logp, 3),
        "HBD": h_donors,
        "HBA": h_acceptors,
        "TPSA": round(tpsa, 3),
        "RotatableBonds": rot_bonds,
        "RingCount": ring_count,
        "AromaticRings": aromatic_rings,
        "HeavyAtomCount": heavy_atoms,
        "FractionCSP3": round(fraction_csp3, 3),
        "LipinskiPassed": passed,
        "LipinskiScore": f"{passed}/4",
        "FunctionalGroups": ", ".join(found_groups) if found_groups else "None found",
    }

    if show_molecule:
        display(Draw.MolToImage(mol, size=(300, 300)))

    return pd.DataFrame([result])


analyse_molecule("CCO", name="ethanol")

## 8. Analyse a small molecule panel

The original project used a small set of simple molecules. This remade version keeps that idea but adds more chemically useful examples, including fragrance-related molecules and common drug-like molecules.

In [ ]:
# ============================================================
# Part 10 — Small molecule practice panel
# ============================================================

molecule_examples = [
    {"name": "ethanol", "smiles": "CCO"},
    {"name": "acetic acid", "smiles": "CC(=O)O"},
    {"name": "benzene", "smiles": "c1ccccc1"},
    {"name": "ethyl acetate", "smiles": "CCOC(=O)C"},
    {"name": "ethylamine", "smiles": "CCN"},
    {"name": "propan-2-ol", "smiles": "CC(C)O"},
    {"name": "acetone", "smiles": "CC(=O)C"},
    {"name": "phenol", "smiles": "Oc1ccccc1"},
    {"name": "toluene", "smiles": "Cc1ccccc1"},
    {"name": "aspirin", "smiles": "CC(=O)OC1=CC=CC=C1C(=O)O"},
    {"name": "caffeine", "smiles": "Cn1cnc2c1c(=O)n(C)c(=O)n2C"},
    {"name": "ibuprofen", "smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"},
    {"name": "limonene", "smiles": "CC1=CCC(CC1)C(=C)C"},
    {"name": "linalool", "smiles": "CC(C)=CCCC(C)(O)C=C"},
    {"name": "vanillin", "smiles": "COC1=C(C=CC(=C1)C=O)O"},
    {"name": "eugenol", "smiles": "COC1=CC(=CC=C1O)CC=C"},
    {"name": "geraniol", "smiles": "CC(C)=CCC/C(C)=C/CO"},
    {"name": "citral", "smiles": "CC(C)=CCCC(C)=CC=O"},
]

rows = []

for item in molecule_examples:
    analysed = analyse_molecule(
        item["smiles"],
        name=item["name"],
        show_molecule=False
    )

    if analysed is not None:
        rows.append(analysed)

df_results = pd.concat(rows, ignore_index=True)

print("Analysed molecules:", len(df_results))
df_results

## 9. Sort and inspect molecular properties

Simple sorting can already reveal chemistry trends.

Examples:

- Higher `LogP` usually means more hydrophobic.
- Higher `TPSA`, `HBD` and `HBA` usually indicate more polarity.
- Higher `RotatableBonds` often means more molecular flexibility.
- More rings and aromatic rings often suggest more rigid or aromatic structures.

In [ ]:
# ============================================================
# Part 11 — Sort molecules by selected descriptors
# ============================================================

display_cols = [
    "name", "MolWt", "LogP", "TPSA", "HBD", "HBA",
    "RotatableBonds", "RingCount", "AromaticRings",
    "LipinskiScore", "FunctionalGroups"
]

print("Most hydrophobic molecules by LogP:")
display(df_results[display_cols].sort_values("LogP", ascending=False).head(8))

print("Most polar molecules by TPSA:")
display(df_results[display_cols].sort_values("TPSA", ascending=False).head(8))

## 10. Draw molecules in a grid

`Draw.MolsToGridImage()` is useful for checking a whole molecule panel at once.

In [ ]:
# ============================================================
# Part 12 — Draw molecule grid
# ============================================================

mols = [
    Chem.MolFromSmiles(item["smiles"])
    for item in molecule_examples
]

legends = [
    item["name"]
    for item in molecule_examples
]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=4,
    subImgSize=(250, 220),
    legends=legends
)

display(img)

## 11. Robust handling of invalid SMILES

Real datasets often contain invalid strings, missing values or duplicates. This section shows the minimal cleaning logic that later projects will reuse.

In [ ]:
# ============================================================
# Part 13 — Validate and clean a molecule table
# ============================================================

raw_molecule_table = pd.DataFrame([
    {"name": "ethanol", "smiles": "CCO"},
    {"name": "ethanol duplicate", "smiles": "OCC"},
    {"name": "acetic acid", "smiles": "CC(=O)O"},
    {"name": "invalid example", "smiles": "ABCXYZ"},
    {"name": "missing example", "smiles": None},
    {"name": "benzene", "smiles": "c1ccccc1"},
])

def safe_mol_from_smiles(smiles):
    """Return RDKit Mol or None for invalid/missing SMILES."""

    if pd.isna(smiles):
        return None

    try:
        return Chem.MolFromSmiles(str(smiles))
    except Exception:
        return None


clean_df = raw_molecule_table.copy()
clean_df["mol"] = clean_df["smiles"].apply(safe_mol_from_smiles)

before_invalid_filter = len(clean_df)
clean_df = clean_df[clean_df["mol"].notnull()].reset_index(drop=True)
after_invalid_filter = len(clean_df)

clean_df["canonical_smiles"] = clean_df["mol"].apply(
    lambda m: Chem.MolToSmiles(m, canonical=True)
)

before_dedup = len(clean_df)
clean_df = clean_df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)
after_dedup = len(clean_df)

print("Removed invalid or missing rows:", before_invalid_filter - after_invalid_filter)
print("Removed duplicate molecules:", before_dedup - after_dedup)

clean_df[["name", "smiles", "canonical_smiles"]]

## 12. Batch analysis function for a DataFrame

This function accepts a table with `name` and `smiles` columns, validates molecules, removes duplicates and returns descriptor results.

In [ ]:
# ============================================================
# Part 14 — Batch analyser for molecule tables
# ============================================================

def analyse_molecule_table(input_df, name_col="name", smiles_col="smiles"):
    """Clean and analyse a molecule table containing names and SMILES strings."""

    required_cols = [name_col, smiles_col]
    missing_cols = [col for col in required_cols if col not in input_df.columns]

    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    work_df = input_df[required_cols].copy()
    work_df["mol"] = work_df[smiles_col].apply(safe_mol_from_smiles)
    work_df = work_df[work_df["mol"].notnull()].reset_index(drop=True)

    work_df["canonical_smiles"] = work_df["mol"].apply(
        lambda m: Chem.MolToSmiles(m, canonical=True)
    )

    work_df = work_df.drop_duplicates(subset="canonical_smiles").reset_index(drop=True)

    result_rows = []

    for _, row in work_df.iterrows():
        analysed = analyse_molecule(
            row["canonical_smiles"],
            name=row[name_col],
            show_molecule=False
        )

        if analysed is not None:
            result_rows.append(analysed)

    if not result_rows:
        return pd.DataFrame()

    return pd.concat(result_rows, ignore_index=True)


clean_results = analyse_molecule_table(raw_molecule_table)
clean_results

## 13. Add a simple fragrance-likeness sanity flag

This is **not** a trained model. It is a rough rule-based sanity check for perfume-related practice. It helps distinguish volatile, moderately hydrophobic organic molecules from highly polar/non-volatile compounds.

Example rough conditions:

- Molecular weight between 80 and 350
- LogP between 1 and 6
- TPSA below 90
- H-bond donors no more than 2

These thresholds are deliberately simple and should be treated as a learning heuristic, not a scientific classifier.

In [ ]:
# ============================================================
# Part 15 — Rule-based fragrance-likeness flag
# ============================================================

def fragrance_likeness_flag(row):
    """Return a rough fragrance-like flag based on simple global descriptors."""

    return (
        80 <= row["MolWt"] <= 350
        and 1.0 <= row["LogP"] <= 6.0
        and row["TPSA"] <= 90
        and row["HBD"] <= 2
    )


df_results["FragranceLikeHeuristic"] = df_results.apply(fragrance_likeness_flag, axis=1)

df_results[
    [
        "name", "MolWt", "LogP", "TPSA", "HBD", "HBA",
        "FunctionalGroups", "FragranceLikeHeuristic"
    ]
].sort_values("FragranceLikeHeuristic", ascending=False)

## 14. Export results

Saving clean descriptor tables creates a reusable dataset for later projects. Project 2 can load the same style of table and use descriptors as machine-learning features.

In [ ]:
# ============================================================
# Part 16 — Export analysed molecules
# ============================================================

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / "project1_rdkit_molecule_analysis.csv"

df_results.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(df_results))
print("Columns:", len(df_results.columns))

## 15. Practice checklist

Use this notebook to confirm you can:

1. Convert SMILES into RDKit `Mol` objects.
2. Detect invalid SMILES safely.
3. Draw one molecule and multiple molecules.
4. Calculate descriptors such as `MolWt`, `LogP`, `TPSA`, `HBD`, `HBA` and rotatable bonds.
5. Apply Lipinski Rule-of-Five checks.
6. Detect functional groups using SMARTS patterns.
7. Analyse multiple molecules in a DataFrame.
8. Export a cleaned descriptor table for later machine-learning projects.

**Link to Project 2**

Project 2 starts from this idea:

```text
molecule table → descriptors → machine-learning features → QSAR model
```